In [ ]:
# =============================================================================
# Exercise 3.45 — Symbolic Solution
# =============================================================================

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Math, HTML

# Disable scrolling for large outputs in Jupyter Notebook
display(HTML("<style>.output_scroll { max-height: none !important; }</style>"))

sp.init_printing(use_unicode=True)
z, n, k = sp.symbols('z n k', complex=True)

print("=== LTI System Response Analysis — Problem 090605 ===")
print()

# =============================================================================
# Given transfer function
# =============================================================================
display(Math(r"\mathbf{\text{Given system}}"))

H = sp.factor((z**(-1) + sp.Rational(1, 2)*z**(-2)) / (1 - sp.Rational(3, 5)*z**(-1) + sp.Rational(2, 25)*z**(-2)))
display(Math(r"\mathcal{H}(z)=" + sp.latex(H)))

H_rational = sp.factor(sp.together(H))
display(Math(r"\mathcal{H}(z)=" + sp.latex(H_rational)))

# =============================================================================
# Poles of the system
# =============================================================================
display(Math(r"\mathbf{\text{Poles}}"))

D = 1 - sp.Rational(3, 5)*z**(-1) + sp.Rational(2, 25)*z**(-2)
poles = sp.solve(sp.Eq(sp.together(D), 0), z)

display(Math(r"D(z)=" + sp.latex(D)))
display(Math(r"\text{Factored denominator: }" + sp.latex(sp.factor(D))))
display(Math(r"\text{Poles: }" + ", ".join(sp.latex(p) for p in poles)))

# =============================================================================
# (a) Impulse response
# =============================================================================
display(Math(r"\mathbf{\text{(a) Impulse response}}"))

Y1 = sp.factor(sp.simplify(z * H))
display(Math(r"\mathcal{H}(z)=z^{-1}\mathcal{Y}_1(z)"))
display(Math(r"\mathcal{Y}_1(z)=" + sp.latex(Y1)))

Fh = sp.factor(Y1 * z**(n - 1))
display(Math(r"\mathcal{Y}_1(z)z^{n-1}=" + sp.latex(Fh)))

res_h = [sp.factor(sp.simplify(sp.residue(Fh, z, p))) for p in poles]
for p, r in zip(poles, res_h):
    display(Math(r"\operatorname{Res}_{z=" + sp.latex(p) + r"}=" + sp.latex(r)))

h1 = sp.factor(sp.simplify(sum(res_h)))
display(Math(r"h_1[n]=" + sp.latex(h1)))

h = sp.factor(sp.simplify(h1.subs(n, n - 1)))
display(Math(r"h[n]=\left(" + sp.latex(h) + r"\right)u[n-1]"))

# =============================================================================
# (b) Zero-state response for x[n] = u[n]
# =============================================================================
display(Math(r"\mathbf{\text{(b) Zero-state response}}"))
display(Math(r"x[n]=u[n]"))

k = sp.symbols('k', integer=True, nonnegative=True)
yzs_conv = sp.factor(sp.simplify(sp.summation(h1.subs(n, k), (k, 0, n - 1))))

display(Math(r"y_{ZS}[n]=x[n]*h[n]"))
display(Math(r"y_{ZS}[n]=\sum_{k=0}^{n}h[n-k]"))
display(Math(r"y_{ZS}[n]=\sum_{m=0}^{n-1}h_1[m]"))
display(Math(r"y_{ZS}[n]=" + sp.latex(yzs_conv)))

yzs_conv_alt = sp.factor(sp.simplify(sp.summation(h1.subs(n, n - k - 1), (k, 0, n - 1))))
display(Math(r"y_{ZS,\mathrm{conv}}[n]=" + sp.latex(yzs_conv_alt)))

convolution_check = sp.simplify(yzs_conv - yzs_conv_alt)
display(Math(r"\text{Convolution verification: }\quad " + sp.latex(convolution_check)))

# =============================================================================
# (c) Step response with initial conditions
# =============================================================================
display(Math(r"\mathbf{\text{(c) Step response with initial conditions}}"))

display(Math(r"\mathcal{H}(z)=" + r"\frac{\mathcal{Y}(z)}{\mathcal{X}(z)}"))
display(Math(r"y[n]-\frac{3}{5}y[n-1]+\frac{2}{25}y[n-2]=x[n-1]+\frac{1}{2}x[n-2]"))

ym1 = sp.Integer(1)
ym2 = sp.Integer(2)
display(Math(r"y[-1]=" + sp.latex(ym1) + r",\qquad y[-2]=" + sp.latex(ym2)))

Yp = sp.Symbol("Y^+(z)")
Yshift1 = z**(-1) * Yp + ym1
Yshift2 = z**(-2) * Yp + z**(-1) * ym1 + ym2

display(Math(r"\mathcal{Z}^{+}\{y[n-1]\}=" + sp.latex(Yshift1)))
display(Math(r"\mathcal{Z}^{+}\{y[n-2]\}=" + sp.latex(Yshift2)))

equation = sp.expand(Yp - sp.Rational(3, 5) * Yshift1 + sp.Rational(2, 25) * Yshift2)
display(Math(r"\text{Homogeneous equation: }" + sp.latex(equation) + r"=0"))

Yzi = sp.factor(sp.solve(sp.Eq(equation, 0), Yp)[0])
display(Math(r"\mathcal{Y}_{ZI}(z)=" + sp.latex(Yzi)))

poles_zi = sp.solve(sp.Eq(sp.denom(sp.together(Yzi)), 0), z)
display(Math(r"\text{Poles: }" + ", ".join(sp.latex(p) for p in poles_zi)))

Fzi = sp.factor(Yzi * z**(n - 1))
display(Math(r"\mathcal{A}(z)=" + sp.latex(Fzi)))

res_zi = [sp.factor(sp.simplify(sp.residue(Fzi, z, p))) for p in poles_zi]
for p, r in zip(poles_zi, res_zi):
    display(Math(r"\operatorname{Res}_{z=" + sp.latex(p) + r"}=" + sp.latex(r)))

yzi = sp.factor(sp.simplify(sum(res_zi)))
display(Math(r"y_{ZI}[n]=" + sp.latex(yzi)))

# =============================================================================
# Total response
# =============================================================================
display(Math(r"\mathbf{\text{Total response}}"))

ytotal = sp.factor(sp.simplify(yzs_conv + yzi))
display(Math(r"y[n]=y_{ZS}[n]+y_{ZI}[n]"))
display(Math(r"y[n]=" + sp.latex(ytotal)))

# =============================================================================
# 5. NUMERICAL EVALUATION & PLOTTING (DISCRETE SIGNAL STEM PLOT)
# =============================================================================
display(Math(r"\mathbf{\text{Zero-State Response Visualization}}"))

def evaluate_symbolic(expr, n_values):
    return np.array([float(sp.re(sp.N(expr.subs(n, int(k))))) for k in n_values])

out = widgets.Output()

def plot_system_response(N=20):
    with out:
        out.clear_output(wait=True)
        n_vec = np.arange(0, N)
        zs_vals = evaluate_symbolic(yzs_conv, n_vec)

        fig, axes = plt.subplots(1, 1, figsize=(10, 4))

        markerline, stemlines, baseline = axes.stem(n_vec, zs_vals, basefmt="k-")
        plt.setp(markerline, markersize=6, markerfacecolor='red', markeredgecolor='red')
        plt.setp(stemlines, linewidth=1.5, color='blue')

        axes.set_title(r"Zero-State Response $y_{ZS}[n]$ for Unit Step Input (Discrete Signal)", fontsize=11, fontweight='bold')
        axes.set_xlabel(r"$n$ (Time Index)")
        axes.set_ylabel(r"$y_{ZS}[n]$")
        axes.grid(True, linestyle='--', alpha=0.6)

        plt.tight_layout()
        plt.show()

plot_system_response()
display(out)